# Captum Insights: Interactive Attribution Visualization

**Learning objectives:**
- Configure `AttributionVisualizer` for an image classification model
- Build a dataset of `Batch` objects from real and synthetic images
- Launch the Insights Flask server and navigate the UI
- Compare attribution methods interactively (IG, Saliency, GradCAM)
- Understand the text classification setup for Insights

**Estimated time:** 15 minutes

**Note:** This notebook requires `captum[insights]` — install with `pip install captum flask flask-compress` if needed.

In [ ]:
import torch
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from PIL import Image, ImageDraw
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights

from captum.insights import AttributionVisualizer, Batch
from captum.insights.attr_vis.features import ImageFeature

print('Imports ready.')
print(f'PyTorch version: {torch.__version__}')

## 1. Load ResNet-18

We use a pretrained ResNet-18 (ImageNet-1K) as the model to explain.

In [ ]:
weights = ResNet18_Weights.IMAGENET1K_V1
model = resnet18(weights=weights)
model.eval()

# First 20 ImageNet class names for display
CLASS_NAMES = [
    'tench', 'goldfish', 'great white shark', 'tiger shark', 'hammerhead',
    'electric ray', 'stingray', 'cock', 'hen', 'ostrich',
    'brambling', 'goldfinch', 'house finch', 'junco', 'indigo bunting',
    'robin', 'bulbul', 'jay', 'magpie', 'chickadee',
]

print(f'ResNet-18 loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Classes configured: {len(CLASS_NAMES)}')

## 2. Configure Preprocessing and Feature Descriptor

`ImageFeature` wraps both the input preprocessing (normalization) and the baseline function (black image).

In [ ]:
# Standard ImageNet normalization
normalize = T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])

# Full preprocessing pipeline
transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    normalize,
])

def baseline_func(input_tensor):
    """Black image baseline: all zeros in normalized space."""
    return torch.zeros_like(input_tensor)

def score_func(model_output):
    """Convert raw logits to probabilities for Insights."""
    return torch.softmax(model_output, dim=1)

# Feature descriptor: tells Insights how to preprocess and display the input
image_feature = ImageFeature(
    name='Input Image',
    baseline_transforms=[baseline_func],
    input_transforms=[transform],
)

print('ImageFeature configured.')
print('  baseline_transforms: [black image baseline]')
print('  input_transforms: [Resize(256) → CenterCrop(224) → ToTensor → Normalize]')

## 3. Build a Synthetic Image Dataset

We generate synthetic images with class-correlated color patterns. Each image has a dominant color channel tied to its label. This gives the attribution methods something meaningful to explain.

In [ ]:
def make_synthetic_image(label: int, size: int = 256) -> Image.Image:
    """
    Create a synthetic image with label-correlated visual features.
    - Label 0-6: color-tinted background with centered geometric shape
    - Shape type and color encode the label
    """
    # Base color varies by label (hue cycling)
    hue_shift = (label * 18) % 180  # 0–180 degrees
    r = int(128 + 100 * np.cos(np.radians(hue_shift)))
    g = int(128 + 100 * np.cos(np.radians(hue_shift + 120)))
    b = int(128 + 100 * np.cos(np.radians(hue_shift + 240)))

    # Noisy background
    arr = np.random.randint(30, 80, (size, size, 3), dtype=np.uint8)
    arr[:, :, 0] = np.clip(arr[:, :, 0] + r // 3, 0, 255)
    arr[:, :, 1] = np.clip(arr[:, :, 1] + g // 3, 0, 255)
    arr[:, :, 2] = np.clip(arr[:, :, 2] + b // 3, 0, 255)

    img = Image.fromarray(arr)
    draw = ImageDraw.Draw(img)

    # Draw centered shape — shape type encodes label
    cx, cy = size // 2, size // 2
    shape_color = (r, g, b)
    shape_radius = size // 4

    if label % 3 == 0:  # Circle
        draw.ellipse(
            [cx - shape_radius, cy - shape_radius,
             cx + shape_radius, cy + shape_radius],
            fill=shape_color
        )
    elif label % 3 == 1:  # Rectangle
        draw.rectangle(
            [cx - shape_radius, cy - shape_radius,
             cx + shape_radius, cy + shape_radius],
            fill=shape_color
        )
    else:  # Triangle
        draw.polygon(
            [(cx, cy - shape_radius),
             (cx - shape_radius, cy + shape_radius),
             (cx + shape_radius, cy + shape_radius)],
            fill=shape_color
        )

    return img


def build_insights_dataset(n_batches: int = 30, n_classes: int = 20):
    """
    Build a list of Batch objects for Captum Insights.
    Each Batch wraps a single example: (input_tensor, label_tensor).
    """
    # Preprocessing for Batch inputs (same as ImageFeature.input_transforms)
    tfm = T.Compose([
        T.Resize(256),
        T.CenterCrop(224),
        T.ToTensor(),
        normalize,
    ])

    batches = []
    for i in range(n_batches):
        label = i % n_classes
        img = make_synthetic_image(label, size=256)
        img_tensor = tfm(img).unsqueeze(0)  # (1, 3, 224, 224)
        label_tensor = torch.tensor([label])
        batches.append(Batch(inputs=img_tensor, labels=label_tensor))

    return batches


# Build dataset
dataset = build_insights_dataset(n_batches=30, n_classes=len(CLASS_NAMES))

print(f'Dataset built: {len(dataset)} batches')
print(f'Input shape per batch: {dataset[0].inputs.shape}')
print(f'Label range: {min(b.labels.item() for b in dataset)} - {max(b.labels.item() for b in dataset)}')

## 4. Preview Generated Images

Verify the synthetic dataset before launching Insights.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes = axes.flatten()

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

for i in range(10):
    img_tensor = dataset[i].inputs.squeeze(0)
    # Denormalize for display
    img_display = (img_tensor * std + mean).clamp(0, 1)
    img_np = img_display.permute(1, 2, 0).numpy()

    label = dataset[i].labels.item()
    axes[i].imshow(img_np)
    axes[i].set_title(f'Label: {label}\n{CLASS_NAMES[label]}', fontsize=8)
    axes[i].axis('off')

plt.suptitle('Synthetic Dataset Preview (first 10 examples)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('insights_dataset_preview.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved: insights_dataset_preview.png')

## 5. Verify Model Predictions on the Dataset

Check that the model produces varied predictions — not just one class.

In [ ]:
from collections import Counter

predictions = []
confidences = []

with torch.no_grad():
    for batch in dataset:
        logits = model(batch.inputs)
        probs = torch.softmax(logits, dim=1)[0]
        pred_class = probs.argmax().item()
        confidence = probs[pred_class].item()
        predictions.append(pred_class)
        confidences.append(confidence)

pred_counts = Counter(predictions)
print(f'Prediction distribution across {len(dataset)} examples:')
for cls_idx, count in sorted(pred_counts.items()):
    cls_name = CLASS_NAMES[cls_idx] if cls_idx < len(CLASS_NAMES) else f'class_{cls_idx}'
    print(f'  Class {cls_idx:3d} ({cls_name:<20s}): {count} examples')

print(f'\nMean confidence: {np.mean(confidences):.1%}')
print(f'Min confidence:  {np.min(confidences):.1%}')

## 6. Configure and Launch Captum Insights

Build the `AttributionVisualizer` and start the Flask server.

**After running the next cell:** open `http://localhost:5001` in your browser.

**To stop the server:** use Kernel → Interrupt in Jupyter, or press Ctrl+C.

In [ ]:
visualizer = AttributionVisualizer(
    models=[model],
    score_func=score_func,
    classes=CLASS_NAMES,
    features=[image_feature],
    dataset=dataset,
    num_examples=4,   # examples shown per page in the UI
)

print('AttributionVisualizer configured.')
print('Launching Flask server on http://localhost:5001')
print('Navigate to that URL in Chrome or Firefox.')
print()
print('In the UI, try:')
print('  1. Switch attribution method: IG → Saliency → Gradient×Input')
print('  2. Change target class (top of right panel)')
print('  3. Adjust magnitude threshold to filter low-attribution pixels')
print('  4. Click Download to export a PNG')
print()
print('Press Kernel → Interrupt to stop the server.')
print('-' * 60)

# This blocks until the kernel is interrupted
visualizer.serve(debug=False, port=5001)

## 7. Jupyter Widget Alternative

If the Flask server approach is inconvenient (e.g., in a remote environment), use the Jupyter widget. This renders Insights inline in the notebook output.

In [ ]:
# Uncomment to use the widget instead of the Flask server
# from captum.insights.widget import CaptumInsightsWidget
#
# widget = CaptumInsightsWidget(visualizer)
# widget.render()
#
# Requires:
#   jupyter nbextension install --py captum --sys-prefix
#   jupyter nbextension enable captum --py --sys-prefix

print('Widget code is commented out.')
print('Uncomment and run if you prefer inline rendering.')

## 8. Manual Attribution Verification

While the Insights UI shows attributions visually, we can verify the underlying computation manually for one example using the same method Insights uses internally.

In [ ]:
from captum.attr import IntegratedGradients
import matplotlib.cm as cm

# Pick the first example
sample_input = dataset[0].inputs
sample_label = dataset[0].labels.item()

# Get model prediction
with torch.no_grad():
    logits = model(sample_input)
    probs = torch.softmax(logits, dim=1)[0]
    pred_class = probs.argmax().item()

# Compute IG attribution
ig = IntegratedGradients(lambda x: model(x))
baseline = torch.zeros_like(sample_input)

attrs, delta = ig.attribute(
    sample_input, baseline,
    target=pred_class,
    n_steps=50,
    return_convergence_delta=True,
)

print(f'Example label:     {sample_label} ({CLASS_NAMES[sample_label]})')
print(f'Model prediction:  {pred_class} ({CLASS_NAMES[min(pred_class, len(CLASS_NAMES)-1)]})')
print(f'Confidence:        {probs[pred_class]:.1%}')
print(f'Attribution shape: {attrs.shape}')
print(f'Convergence delta: {delta.item():.6f}')

# Unsigned attribution aggregated over channels
attr_map = attrs.abs().sum(dim=1).squeeze(0).detach()

In [ ]:
# Visualize: this is what Insights shows in the UI
mean_t = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std_t  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

img_display = (sample_input.squeeze(0) * std_t + mean_t).clamp(0, 1)
img_np = img_display.permute(1, 2, 0).numpy()

attr_np = attr_map.numpy()
attr_np_norm = (attr_np - attr_np.min()) / (attr_np.max() - attr_np.min() + 1e-8)

heatmap_rgb = cm.hot(attr_np_norm)[:, :, :3]
overlay = 0.6 * img_np + 0.4 * heatmap_rgb

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img_np)
axes[0].set_title('Input Image')
axes[1].imshow(attr_np_norm, cmap='hot')
axes[1].set_title('IG Attribution (|channels|, normalized)')
axes[2].imshow(overlay.clip(0, 1))
axes[2].set_title('Overlay (60% image + 40% heatmap)')
for ax in axes:
    ax.axis('off')

pred_name = CLASS_NAMES[min(pred_class, len(CLASS_NAMES)-1)]
fig.suptitle(
    f'Manual IG Attribution — Prediction: {pred_name} ({probs[pred_class]:.1%})\n'
    f'Convergence delta: {delta.item():.5f}',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig('insights_manual_attr.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved: insights_manual_attr.png')
print('This is the attribution that Captum Insights displays in the UI heatmap.')

## 9. Text Classification Insights Setup (Reference)

The pattern for text models is similar. Here we show the configuration code without launching the server — use this as a template for NLP models.

In [ ]:
# Text Insights setup — reference code (does not run model; shows pattern)

from captum.insights.attr_vis.features import TextFeature

# Suppose we have:
#   bert_model: AutoModelForSequenceClassification
#   tokenizer: AutoTokenizer

# text_feature = TextFeature(
#     name='Input Text',
#     baseline_transforms=[
#         lambda embeds: torch.zeros_like(embeds)  # zero-embedding baseline
#     ],
#     input_transforms=[],   # tokenization handled in dataset construction
#     visualization_transform=lambda ids:
#         tokenizer.convert_ids_to_tokens(ids[0].tolist()),
# )
#
# text_visualizer = AttributionVisualizer(
#     models=[bert_model],
#     score_func=lambda out: torch.softmax(out, dim=1),
#     classes=['NEGATIVE', 'POSITIVE'],
#     features=[text_feature],
#     dataset=text_batch_list,
#     num_examples=4,
# )
# text_visualizer.serve(debug=False, port=5002)

print('Text Insights reference code shown above (commented out).')
print('The pattern is identical to image Insights — only the feature type changes.')
print()
print('Key difference: TextFeature.visualization_transform converts token IDs')
print('back to readable strings (including ## subword prefixes) for the UI.')

## Self-Check Questions

1. **Baseline effect:** Change `baseline_func` to return `torch.ones_like(input_tensor) * 0.5` (gray baseline) instead of zeros. Does the manual attribution heatmap change noticeably? What regions change most?

2. **Score function:** The `score_func` converts logits to probabilities. What happens if you pass raw logits instead (remove `softmax`)? Does Insights show different attributions? Why or why not?

3. **num_examples:** Change `num_examples` from 4 to 8 in the `AttributionVisualizer`. What happens to the UI layout? What is the performance impact?

4. **Method comparison:** In the Insights UI, switch between IG and Saliency for the same example. Do they highlight the same regions? Where do they disagree most?

5. **Synthetic dataset design:** The synthetic images have shape-correlated color and geometry. For which classes do you expect attribution to concentrate on the shape vs. the background? Check this in the UI.